In [175]:
#FULL DOMAIN RUN

# code for tracing particles back to SBZ draft (python version 3.10.9) (not optimized with numpy.where)
import numpy as np
import matplotlib.pyplot as plt
import xarray as xr
import os; import time

start_time = time.time();

#data loading
################################################################################################################################################################################################################
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
data=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_test7tundra-7_062217.nc') #***
times=data['time'].values/(1e9 * 60); times=times.astype(float);
minutes=1/times[1] #1 / minutes per timestep = timesteps per minute
parcel=xr.open_dataset(dir+'../cm1r20.3/run/cm1out_pdata_test5tundra-7_062217.nc') #***
kms=np.argmax(data['xh'].values-data['xh'][0].values >= 1)
res='1km'

#LOADING CL MAXS FROM CL TRACKING ALGORITHM
dir='/mnt/lustre/koa/koastore/torri_group/air_directory/DCI-Project/'
whereCL=xr.open_dataset(dir+'Project_Algorithms/whereCL_4_0622.nc').load()
whereCL=whereCL.isel(time=slice(0,len(data['time'])))
whereCL=whereCL['maxconv_x']
def get_conv_x(position):
    Conv_X_Max=whereCL[position].item()
    return Conv_X_Max
    
#job array things
################################################################################################################################################################################################################
num_jobs=60 #how many total jobs are being run? i.e. array=1-100 ==> num_jobs=100 #***

job_id = int(os.environ.get('SLURM_ARRAY_TASK_ID', 0)) #this is the current SBATCH job id
if job_id==0: job_id=1
num_parcels=len(parcel['xh']) #total number of parcels
job_range = num_parcels//num_jobs #number of parcels per job 

# Calculate start and end based on job_id
start_job = (job_id - 1) * job_range
end_job = start_job + job_range
if job_id==num_jobs: end_job=num_parcels
print(f'running for parcels {start_job}-{end_job-1}')
parcel=parcel.isel(xh=slice(start_job,end_job)) #slices the lagrangian parcel data
index_adjust=parcel['xh'].values.astype(int)[0]-1 #for correction of parcel index name

running for parcels 6249-8331


In [97]:
# Reading Back Data Later
##############
import h5py
dir2=dir+'Project_Algorithms/Lagrangian_Binary_Array/'
with h5py.File(dir2+'lagrangian_binary_array.h5', 'r') as f:
    # Load the dataset by its name
    A_g = f['A_g'][:]
    A_c = f['A_c'][:]

    W = f['W'][:]
    # QCQI = f['QCQI'][:]
    Z = f['Z'][:]
    Y = f['Y'][:]
    X = f['X'][:]

# #Making Time Matrix
# rows, cols = A.shape[0], A.shape[1]
# T = np.arange(rows).reshape(-1, 1) * np.ones((1, cols), dtype=int)

In [ ]:
#################################################################

In [ ]:
#Updated Lagrangian Tracking Algorithm

#Algorithm Steps:
#(1) Find the first time a parcel is above the LFC:
#(2) First check if the parcel ascends (w>=0.1) for another 20 minutes
#(3) If so, find first time, the parcel slows down (w<0.1)
#(4) If that time is when the parcel is above 750m, save it, "forget", and move on to next parcel
#(5) If that time is when the parcel is below 750m, check if it is within 2km of the CL_Max found from the CL Tracking Algorithm
#(6) If the parcel is near the CL, store in, otherwise save it, "forget", and move on to next parcel
#(7) Continue to next parcel

#(Also, if during, traceback, the parcel escapes the x or z boundary, "forget" parcel, and move on)

In [188]:
#Numerical Settings
Nt=len(data['time'])
Np=len(parcel['xh'])
dt=times[1]*60 #300 secs

#For saving ascend-after-LFC info
ascend_lst=[]
CLmaxheight=750 #750m

#BL slow-down-threshold
w_thresh=0.1

#PARCEL NUMBERS
Ps=parcel['xh'].values.astype(int)

In [189]:
# if ((x + dt*u)==0) or ((z + dt*w)==0)
# u=u[t,Z[t,p],Y[t,p],X[t,p]]; W=W[t,p]
# [u[t,Z[t,p],Y[t,p],X[t,p]] for t in time_arr] >np.max(data['xf'].values) or < np.min(data['xf'].values)
# similarly for w
################################################################################################################
#BOUNDARY-ESCAPE CONDITION
xmin=np.min(data['xf'].values)*1e3
xmax=np.max(data['xf'].values)*1e3
zmin=np.min(data['zf'].values)*1e3
zmax=np.max(data['zf'].values)*1e3

def check_boundary(p,where_BL,above_LFC):
    time_arr=np.arange(where_BL,above_LFC)

    def get_u(t,z,y,x):
        return data['u'].isel(time=t,zh=z,yh=y).interp(xf=data['xh']).isel(xh=x).item()
        # return data['uinterp'].isel(time=t,zh=z,yh=y,uh=u).item()
    def get_x(t,p):
        return parcel['x'][t,p].item()
    def get_w(t,z,y,x):
        return data['w'].isel(time=t,yh=y,xh=x).interp(zf=data['zh']).isel(zh=z).item()
        # return data['winterp'].isel(time=t,zh=z,yh=y,uh=u).item()
    def get_z(t,p):
        return parcel['z'][t,p].item()


    x_tend = [get_x(t, p) + dt * get_u(t, z, y, x)  
          for (t, z, y, x) in zip(time_arr, Z[time_arr, p], Y[time_arr, p], X[time_arr, p])] 
    z_tend = [get_z(t, p) + dt * get_w(t, z, y, x)  
          for (t, z, y, x) in zip(time_arr, Z[time_arr, p], Y[time_arr, p], X[time_arr, p])] 

    x_bound=any(val < xmin or val > xmax for val in x_tend)*1
    z_bound=any(val < zmin or val > zmax for val in z_tend)*1

    out=(x_bound,z_bound)
    if any(np.array(out)==1):
        print(f'parcel {p} crossed boundary between t={where_BL} and t={above_LFC}')
    return out
#############################################################################################################

In [190]:
#Initialize Output Storage Vector

#int 32 can store up to the number 2,147,483,647 
#int 32 has 4 bytes per number, so needs (Np*3)*4 bytes of memory
#Np=125000 ==> (125000*3*4)/(1024**3) = 0.001 GB
#Np=50e6 ==> (50e6*3*4)/(1024**3) = 0.56 GB

out_nz=np.zeros((Np,3)) #(Why Did I Call It out_nz I Will Never Know)
save_nz=np.zeros((Np,3)) #This one is for saving continued-ascent, slow-below-750m parcels that are not with 2 km of CL
save2_nz=np.zeros((Np,3)) #This one is for saving continued-ascent, slow-above-750m parcels

In [ ]:
#############################################################################################################

In [ ]:
#1--------------Looping over each parcel
for count,p in enumerate(Ps):
    if np.mod(p,1e4)==0: print(f'current parcel: {p}/{Np}')
    
    W_p = W[:,p] ***
    u *** # LFC_p = LFC[:,p] 
    
    LFC_p=np.array([0,0,0,0,0,0,1,1,1,1,1,1,1]) ***#1 ==> Parcel is >= LFC
    LFC_p***
    
    #----FIND WHERE PARCEL IS ABOVE LFC----
    indices = np.where(LFC_p == 1)[0]; above_LFC = indices[0] if indices.size > 0 else -999; #FIRST TIME ABOVE LFC
    if above_LFC ==-999:
        continue #if the parcel is never above the LFC, skip the parcel
    
    #----CHECK IF ASCENDS FOR >= 20 minutes AFTER LFC----
    ascend_array=W_p[above_LFC+1:]
    indices=np.where(ascend_array==0)[0]; ascend_stop=indices[0] if indices.size > 0 else 10000; #location of where parcel stops ascending (labeled 10000 to mark for future analysis)
    ascend_lst.append(ascend_stop) #(also store for histogram)
    if ascend_stop>=20*minutes:
    
        #----FIND THE FIRST TIME W_p<=w_thresh----
        indices=np.where(W_p[0:above_LFC]<w_thresh)[0]
        where_BL==indices[-1] if indices.size > 0 else -999 #FIRST PRIOR TIME W<0.1 (IN THE BL)
        if where_BL ==-999:
            continue #if the parcel never slows down backwards in time (unlikely), skip the parcel
            
        #check for boundary escapes
        ################################
        future_location=check_boundary(p,where_BL,above_LFC)
        if (future_location[0]+future_location[1]>=1): continue #if parcel crosses boundary, skips current parcel
        ################################
        
        #----CHECK IF PARCEL SLOWED DOWN LOW ENOUGH----
        if parcel['z'][where_BL].values<=CLmaxheight*kms/1000: #PARCEL MUST BE BELOW 750m WHEN CONTACTING CL
    
            #----CHECK IF CL IS WITHIN 2km----
            #Find the CL-max x-location
            position=(where_BL,Z[where_BL,p],Y[where_BL,p],X[where_BL,p]) #Get the parcels position
            CONV_X=get_conv(position)
            if CONV_X==-1:
                #SAVE PARCEL
                save_nz[p,0]=p
                save_nz[p,1]=where_BL
                save_nz[p,2]=above_LFC 
                continue #if CL doesn't exist along y, save and skip the parcel
            
            if X[where_BL,p] in [CONV_X-2*kms,CONV_X+2*kms]:
                #save X's (t,p) 
                out_nz[p,0]=p
                out_nz[p,1]=where_BL
                out_nz[p,2]=above_LFC 
            else: #continued-ascent, slow-below-750m parcels that are not with 2 km of CL
                #SAVE PARCEL
                save_nz[p,0]=p
                save_nz[p,1]=where_BL
                save_nz[p,2]=above_LFC 
    
        else: #continued-ascent, slow-above-750m parcels
            #SAVE PARCEL
            save2_nz[p,0]=p
            save2_nz[p,1]=where_BL
            save2_nz[p,2]=above_LFC         
            
        #END OF LOOP, THEN WE MOVE ON TO NEXT PARCEL p

In [ ]:
#Storing output and save data
###################################################################################################################################
out_arr[np.where(np.any(out_arr != 0, axis=1))[0],0]+=index_adjust #*needed for job array*+=index_adjust #*needed for job array*
save_arr[np.where(np.any(save_arr != 0, axis=1))[0],0]+=index_adjust #*needed for job array*+=index_adjust #*needed for job array*
ds=xr.Dataset({
    'out_arr': (['rows', 'columns'], out_arr.astype(float)),
    'save_arr': (['rows', 'columns'], save_arr.astype(float)),
    'save2_arr': (['rows', 'columns'], save_arr.astype(float)),
})
ds.to_netcdf(dir+'Project_Algorithms/Tracking_Algorithms/trackout/parcel_tracking'+str(job_id)+'.nc') #*needed for job array*
# ds.to_netcdf(dir+'tracking_algorithms/trackout/SBZlimited_parcel_tracking'+str(job_id)+'.nc')
###########################################################################################################################